In [25]:
import pdfplumber
import pathlib
import logging
import re
import pandas as pd
from pathlib import Path
from dataclasses import dataclass

In [26]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

Data Class

In [27]:
@ dataclass
class Transaction:
    date:        str
    description: str
    amount:      float
    raw_text:    str = ""
    source_tier: str = ""

In [28]:
_ACCOUNT_SIGNALS = {
    "credit": [
        re.compile(r"minimum\s+payment\s+due",         re.I),
        re.compile(r"credit\s+limit",                  re.I),
        re.compile(r"available\s+credit",              re.I),
        re.compile(r"payments.{0,10}credits.{0,10}adjustments", re.I),
        re.compile(r"statement\s+balance",             re.I),
        re.compile(r"cash\s+advance",                  re.I),
        re.compile(r"purchase\s+apr",                  re.I),
        re.compile(r"rewards?\s+points?",              re.I),
    ],
    "checking": [
        re.compile(r"deposits?\s+and\s+additions",     re.I),
        re.compile(r"checks?\s+paid",                  re.I),
        re.compile(r"direct\s+deposit",                re.I),
        re.compile(r"\boverdraft\b",                   re.I),
        re.compile(r"debit\s+card\s+purchases?",       re.I),
        re.compile(r"atm\s+withdrawal",                re.I),
        re.compile(r"checking\s+account",              re.I),
    ],
    "savings": [
        re.compile(r"interest\s+earned",               re.I),
        re.compile(r"annual\s+percentage\s+yield",     re.I),
        re.compile(r"\bapy\b",                         re.I),
        re.compile(r"interest\s+(paid|credited)",      re.I),
        re.compile(r"online\s+savings",                re.I),
        re.compile(r"high.yield\s+savings",            re.I),
        re.compile(r"savings\s+account",               re.I),
        re.compile(r"money\s+market",                  re.I),
    ],
}


def classify_account_type(text: str) -> str | None:
    """Classify account type by scoring regex signal hits across the full statement text."""
    scores = {account_type: 0 for account_type in _ACCOUNT_SIGNALS}
    for account_type, patterns in _ACCOUNT_SIGNALS.items():
        for pattern in patterns:
            if pattern.search(text):
                scores[account_type] += 1

    best_type, best_score = max(scores.items(), key=lambda x: x[1])
    return best_type if best_score > 0 else None

In [29]:
def extract_text(pdf_path: str) -> str:
    #Text transaction from PDF
    #logger.info("Tier 1: pdfplumber text extraction")
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    return "\n".join(pages_text)

In [30]:
_DATE_PATTERN = (
    r'\b\d{1,2}/\d{1,2}(?:/\d{2,4})?\b|'
    r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|'
    r'May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|'
    r'Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{1,2}\b'
)
_AMOUNT_PATTERN = r'-?\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'

_TRANSACTION_ROW = re.compile(
    rf'^\s*({_DATE_PATTERN})'
    rf'(?:\s+({_DATE_PATTERN}))?'
    rf'\s+(.*?)'
    rf'\s+({_AMOUNT_PATTERN})'
    rf'(?:\s+({_AMOUNT_PATTERN}))?'
    rf'\s*$',
    re.I,
)

_NEGATIVE_SECTION_KEYWORDS = ["PAYMENTS, CREDITS AND ADJUSTMENTS"]

_CC_PAYMENT_PATTERN = re.compile(
    r'\b(autopay|pymt|payment|ach\s+transfer)\b',
    re.I,
)


def extract_transactions(bank_text: str) -> pd.DataFrame:
    """Parse transaction rows from extracted PDF text into a DataFrame."""
    transactions = []
    is_negative_section = False

    for line in bank_text.splitlines():
        line = " ".join(line.split())
        if not line:
            continue

        if any(keyword in line.upper() for keyword in _NEGATIVE_SECTION_KEYWORDS):
            is_negative_section = True
            continue
        elif "TRANSACTIONS" in line.upper():
            is_negative_section = False
            continue

        match = _TRANSACTION_ROW.match(line)
        if match:
            trans_date, post_date, description, amount1, amount2 = match.groups()
            description = description.strip()

            # Skip CC payment rows - these are transfers, not expenses
            if is_negative_section and _CC_PAYMENT_PATTERN.search(description):
                continue

            amount1_num = float(amount1.replace("$", "").replace(",", ""))
            amount2_num = float(amount2.replace("$", "").replace(",", "")) if amount2 else None

            if is_negative_section:
                amount1_num = -abs(amount1_num)
                if amount2_num is not None:
                    amount2_num = -abs(amount2_num)

            transactions.append({
                "trans_date":  trans_date,
                "post_date":   post_date,
                "description": description,
                "amount1":     amount1_num,
                "amount2":     amount2_num,
            })

    return pd.DataFrame(transactions)

In [31]:
_DATE_PAT = (
    r'\d{1,2}/\d{1,2}/\d{2,4}'
    r'|(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?'
    r'|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\s+\d{1,2},?\s+\d{4}'
)

_PERIOD_RE = re.compile(rf'({_DATE_PAT})\s*(?:through|to|[-–])\s*({_DATE_PAT})', re.I)


def extract_statement_period(text: str) -> dict:
    """Return {'period_start': 'YYYY-MM-DD', 'period_end': 'YYYY-MM-DD'} or Nones."""
    m = _PERIOD_RE.search(text[:3000])
    if m:
        try:
            return {
                "period_start": pd.to_datetime(m.group(1)).strftime("%Y-%m-%d"),
                "period_end":   pd.to_datetime(m.group(2)).strftime("%Y-%m-%d"),
            }
        except Exception:
            pass
    return {"period_start": None, "period_end": None}

Main Functions

In [32]:
PDFS = {
    "Capital_One_Venture": "../data/Capital_One_102025_2952.pdf"
    # "Chase_Sapphire":      "../data/samples/Chase_Sapphire_20260317-statements-1333-.pdf",
    # "Marcus_Saving":       "../data/samples/Marcus_STMTCMB100_20260401_9279_Jiang_1529495_98986.PDF",
    # "Chase_College":       "../data/samples/Chase_College_20260325-statements-8585-.pdf",
    # #"Chase_Year_end":       "../data/samples/Spending Report PDF.pdf", # This doesn't work because not statement format
    # "Chase_Sapphire_Dec":    "../data/samples/Chase_Sapphire_20251217-statements-1333-.pdf",
    # 'Bofa':                 "../data/samples/eStmt_2025-09-06.pdf",
    # 'Bofa_paid_debt':       "../data/samples/bofa_paid_debt.pdf"
}


In [33]:
def parse_pdf(pdf_path: str) -> dict:
    """
    Parse a bank PDF statement into transactions + metadata.

    Returns
    -------
    {
        "period_start": "YYYY-MM-DD" or None,
        "period_end":   "YYYY-MM-DD" or None,
        "account_type": "credit" | "checking" | "savings" | None,
        "transactions": pd.DataFrame,
    }
    """
    text = extract_text(pdf_path)
    df = extract_transactions(text)
    period = extract_statement_period(text)
    account_type = classify_account_type(text)

    logger.info(
        "Parsed %s: %d transactions, period %s → %s, account_type %s",
        Path(pdf_path).name,
        len(df),
        period["period_start"],
        period["period_end"],
        account_type,
    )

    return {
        "period_start": period["period_start"],
        "period_end":   period["period_end"],
        "account_type": account_type,
        "transactions": df,
    }


In [34]:
parse_pdf(PDFS['Capital_One_Venture'])

INFO | Parsed Capital_One_102025_2952.pdf: 73 transactions, period 2025-09-20 → 2025-10-20, account_type credit


{'period_start': '2025-09-20',
 'period_end': '2025-10-20',
 'account_type': 'credit',
 'transactions':    trans_date post_date                       description  amount1 amount2
 0      Oct 15    Oct 16           aliexpressSan MateoCA -    -7.89    None
 1      Oct 17    Oct 18   GETYOURGUIDE TICKETSLONDONGBR -  -109.79    None
 2      Oct 17    Oct 18   GETYOURGUIDE TICKETSLONDONGBR -   -34.41    None
 3      Sep 19    Sep 20        WWW.VOXI.CO.UKVODAFONE LTD    13.73    None
 4      Sep 21    Sep 22  SumUp *fresh meetcanning townGBR    15.59    None
 ..        ...       ...                               ...      ...     ...
 68     Oct 18    Oct 20         SQ *BREAD AHEAD LTDLondon     6.06    None
 69     Oct 18    Oct 20      TIAN TIAN MARKET - CANLONDON    66.58    None
 70     Oct 18    Oct 20       ASDA SUPERSTOREISLE OF DOGS    18.38    None
 71     Oct 18    Oct 20        TFL TRAVEL CHTFL.GOV.UK/CP     3.91    None
 72     Oct 18    Oct 20               MARUGAME UDONLONDON   